# Fine-Tuning BERT for Multi-Class Text Classification

### Introduction

This notebook demonstrates the process of fine-tuning a pre-trained BERT model (`bert-base-uncased`) for a multi-class text classification task with 27 classes. The workflow covers all essential steps, from dataset preparation to model evaluation, using the Hugging Face Transformers library.

**Key steps in this notebook include:**

1. **Label Mapping**  
   - Define consistent mappings between string labels and numerical IDs (`label2id` and `id2label`) for training and evaluation.

2. **Dataset Preparation**  
   - Extract the dataset from a ZIP archive and load it using the Hugging Face `datasets` library.  
   - Tokenize the text using the BERT tokenizer and prepare it for model input.  
   - Use a data collator for dynamic padding during training.

3. **Model Setup and Fine-Tuning**  
   - Load a pre-trained BERT model for sequence classification.  
   - Freeze the majority of BERT layers and fine-tune the last two transformer layers along with the pooler and classifier head to optimize performance efficiently.  
   - Implement **Layer-wise Learning Rate Decay (LLRD)** to train different layers at different learning rates, improving stability and convergence.

4. **Training**  
   - Use the Hugging Face `Trainer` API to manage training, evaluation, checkpointing, and early stopping.  
   - Monitor validation metrics and save the best-performing model.

5. **Model Saving and Archiving**  
   - Save the fine-tuned model and tokenizer.  
   - Create a ZIP archive for easy sharing or deployment.

6. **Evaluation**  
   - Generate predictions on the test dataset.  
   - Compute a detailed **classification report** and visualize a normalized **confusion matrix** to analyze per-class performance.

This notebook provides a full, end-to-end workflow for fine-tuning BERT on custom text classification tasks, leveraging advanced techniques such as selective layer unfreezing and LLRD to achieve efficient and effective model training.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

import torch

from datasets import load_dataset
from datasets import Dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from transformers import BertForSequenceClassification, BertTokenizer
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import EarlyStoppingCallback

from evaluate import load
import os
import zipfile

### Extracting the Dataset from a ZIP File

This code snippet handles the extraction of the dataset from a ZIP archive:

1. **`zip_file_path`** specifies the path to the ZIP file containing the dataset.  
2. **`extract_to`** specifies the directory where the files will be extracted.  
3. The script checks if the target directory exists; if not, it creates it using `os.makedirs()`.  
4. The `zipfile.ZipFile` context manager is used to open the ZIP file in read mode, and `extractall()` extracts all files into the specified directory.

This ensures that the dataset is available in an accessible folder for further preprocessing and model training.

In [ ]:
zip_file_path = "ds_cl.zip"
extract_to = "./ds_cl"

if not os.path.exists(extract_to):
    os.makedirs(extract_to)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

### Mapping Between Labels and IDs

In this section, we define two dictionaries to map between the textual class labels and their corresponding numerical IDs. These mappings are essential for text classification tasks because models work with numerical labels internally, while the original data may have string-based class identifiers.

- **`id2label`**: Converts a numerical ID (used by the model) back to its original string label. This is useful when interpreting the model's predictions.  
- **`label2id`**: Converts a string label from the dataset to its corresponding numerical ID. This is used when encoding the dataset before training.

Having consistent `id2label` and `label2id` mappings ensures that the model outputs and evaluation metrics are correctly aligned with the original class labels.

In [ ]:
id2label = {
    0: "10",
    1: "40",
    2: "50",
    3: "60",
    4: "1140",
    5: "1160",
    6: "1180",
    7: "1280",
    8: "1281",
    9: "1300",
    10: "1301",
    11: "1302",
    12: "1320",
    13: "1560",
    14: "1920",
    15: "1940",
    16: "2060",
    17: "2220",
    18: "2280",
    19: "2403",
    20: "2462",
    21: "2522",
    22: "2582",
    23: "2583",
    24: "2585",
    25: "2705",
    26: "2905"
}

label2id = {
    "10": 0,
    "40": 1,
    "50": 2,
    "60": 3,
    "1140": 4,
    "1160": 5,
    "1180": 6,
    "1280": 7,
    "1281": 8,
    "1300": 9,
    "1301": 10,
    "1302": 11,
    "1320": 12,
    "1560": 13,
    "1920": 14,
    "1940": 15,
    "2060": 16,
    "2220": 17,
    "2280": 18,
    "2403": 19,
    "2462": 20,
    "2522": 21,
    "2582": 22,
    "2583": 23,
    "2585": 24,
    "2705": 25,
    "2905": 26
}

### Loading the Dataset, Tokenization, and Preparing the Model

This section covers the steps for loading the dataset, tokenizing the text, and setting up the pre-trained BERT model for sequence classification:

1. **Imports and Device Setup**  
   - Import necessary libraries for dataset handling (`datasets`), model and tokenizer (`transformers`), evaluation (`evaluate`), and tensor operations (`torch`, `numpy`).  
   - Determine the device (`CPU` or `GPU`) for model training.

2. **Loading the Dataset**  
   - Load the preprocessed dataset from disk using `load_from_disk()`.  

3. **Tokenizer Setup and Tokenization**  
   - Load the `bert-base-uncased` tokenizer.  
   - Define `tokenize_function` to tokenize input text, applying truncation and padding to a maximum sequence length of 128.  
   - Use `dataset.map()` to tokenize the entire dataset efficiently in batches.  
   - Initialize `DataCollatorWithPadding` to handle dynamic padding during training.

4. **Model Setup**  
   - Load a pre-trained BERT model for sequence classification with 27 labels and specify `id2label` and `label2id` mappings.  

5. **Layer Freezing and Fine-Tuning**  
   - Freeze all BERT layers initially to prevent updates during training.  
   - Unfreeze the last two transformer layers (`encoder.layer.10` and `encoder.layer.11`) for fine-tuning.  
   - Optionally, unfreeze the pooler and classifier head to allow training on these layers.  

6. **Check Trainable Parameters**  
   - Print the total number of trainable parameters to verify which parts of the model will be updated during training.

This setup allows efficient fine-tuning of the model by training only the most relevant layers while keeping the majority of BERT frozen, reducing memory usage and training time.

In [ ]:
import numpy as np
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from evaluate import load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_from_disk("./ds_cl")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"],
                     padding="max_length",
                     truncation=True,
                     max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=27,
    id2label=id2label,
    label2id=label2id
)

# Freeze entire BERT
for p in model.bert.parameters():
    p.requires_grad = False

# Unfreeze last two transformer layers
for name, param in model.bert.named_parameters():
    if name.startswith("encoder.layer.10") or name.startswith("encoder.layer.11"):
        param.requires_grad = True

# Unfreeze pooler (optional but recommended)
for name, param in model.bert.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

# Unfreeze classifier head
for p in model.classifier.parameters():
    p.requires_grad = True

print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

### Setting Up Layer-wise Learning Rate Decay (LLRD) and Trainer

This section prepares the training setup, including metrics, optimizer configuration with Layer-wise Learning Rate Decay (LLRD), and the Hugging Face `Trainer`.

1. **Evaluation Metrics**  
   - Define `compute_metrics()` to calculate **accuracy** during evaluation by comparing predicted labels with ground truth labels.

2. **Layer-wise Learning Rate Decay (LLRD)**  
   - `get_llrd_params()` constructs parameter groups with different learning rates:  
     - **Classifier head**: highest learning rate.  
     - **Pooler**: moderate learning rate.  
     - **Transformer encoder layers**: progressively smaller learning rates from top (last) layer to bottom (first) layer.  
     - **Embeddings**: smallest learning rate.  
   - This approach allows fine-tuning the model more effectively by updating higher layers more aggressively while keeping lower layers stable.

3. **Training Arguments**  
   - `TrainingArguments` specifies training hyperparameters, such as batch size, number of epochs, learning rate, warmup, weight decay, evaluation strategy, mixed precision (`fp16`), and logging.  
   - `load_best_model_at_end=True` ensures the best model according to validation accuracy is kept.

4. **Trainer Initialization**  
   - `Trainer` handles the training loop, evaluation, and checkpointing.  
   - Pass in the model, datasets, tokenizer, data collator, metrics function, and custom optimizer with LLRD parameter groups.  
   - Use `EarlyStoppingCallback` to stop training if validation accuracy does not improve for 5 consecutive evaluation steps.

This setup leverages advanced fine-tuning techniques like LLRD and early stopping to improve performance while optimizing training stability and efficiency.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np

# -------- Metrics ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": np.mean(preds == labels)}

# --------- LLRD PARAMETER GROUPS ----------
def get_llrd_params(model, base_lr=5e-5):
    # Layer-wise learning rate decay
    lr_factor = 0.85

    opt_params = []
    
    # Classifier head (highest LR)
    opt_params.append({
        "params": model.classifier.parameters(),
        "lr": base_lr * 3.0
    })

    # Pooler (optional)
    opt_params.append({
        "params": model.bert.pooler.parameters(),
        "lr": base_lr
    })

    # Encoder layers: progressively smaller LR
    for layer_idx in range(11, -1, -1):  # layer 11 down → 0
        layer = f"encoder.layer.{layer_idx}."
        params = [p for n, p in model.bert.named_parameters() if n.startswith(layer)]

        if len(params) == 0:
            continue

        layer_lr = base_lr * (lr_factor ** (11 - layer_idx))

        opt_params.append({
            "params": params,
            "lr": layer_lr
        })

    # Embeddings (smallest LR)
    embed_params = list(model.bert.embeddings.parameters())
    opt_params.append({
        "params": embed_params,
        "lr": base_lr * (lr_factor ** 12)
    })

    return opt_params

# --------- Training Arguments ----------
training_args = TrainingArguments(
    output_dir="./birdy-optimized",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-5,             # Base LR (LLRD scales from this)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    warmup_ratio=0.06,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,                      # Or bf16=True
    report_to="none"
)

optimizer_params = get_llrd_params(model, base_lr=5e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(
        torch.optim.AdamW(
            optimizer_params,
            betas=(0.9, 0.98),
            eps=1e-6,
            weight_decay=0.01
        ),
        None  # HF Trainer will create scheduler automatically
    ),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5)
    ]
)


### Training the Model and Saving the Best Checkpoint

This snippet starts the fine-tuning process and saves the trained model:

1. **Start Training**  
   - `trainer.train()` runs the full training loop using the datasets, optimizer, and training arguments defined earlier.  
   - The Hugging Face `Trainer` handles batching, evaluation, checkpointing, and logging automatically.

2. **Save the Model**  
   - `trainer.save_model("./birdy-optim")` saves the fine-tuned model and tokenizer to the specified directory.  
   - This allows you to reload the trained model later for inference or further fine-tuning.

After this step, the model is fully trained and ready for evaluation or deployment.

In [ ]:
trainer.train()
trainer.save_model("./birdy-optim")

### Creating a ZIP Archive of the Trained Model

This snippet compresses the fine-tuned model into a ZIP file for easy storage or sharing:

1. **Model Path**  
   - `path = "./birdy-optim"` points to the directory containing the saved model and tokenizer.

2. **Create ZIP Archive**  
   - `shutil.make_archive(path, 'zip', path)` creates a ZIP file named `birdy-optim.zip` containing all files in the model directory.  

This allows the trained model to be easily uploaded, transferred, or backed up.

In [ ]:
import shutil
path = "./birdy-optim"
shutil.make_archive(path, 'zip', path)

### Evaluation: Classification Report and Confusion Matrix

This section defines helper functions to evaluate and visualize the model's performance:

1. **Load Label Mapping**  
   - `load_id2label()` loads the `id2label` mapping from a JSON file.  
   - `id2_label()` converts a numeric class ID into its corresponding string label.

2. **Display Metrics and Confusion Matrix**  
   - `display_cm(labels, predicted_classes)` prints the **classification report** (precision, recall, F1-score, support) using `sklearn.metrics.classification_report`.  
   - Computes the **normalized confusion matrix** to show how predictions compare to true labels.  
   - Uses `seaborn` to create a heatmap of the confusion matrix for easy visualization, with class labels on both axes.  
   - Normalization and rounding make the matrix easier to interpret.

These functions allow a detailed analysis of the model’s predictions across all classes, helping identify strengths and weaknesses in classification performance.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

# ---------------------------------------------------------
# DISPLAY FUNCTION FROM YOU
# ---------------------------------------------------------
def load_id2label(file_path="./id2label"):

    with open(file_path, "r", encoding="utf-8") as f:
        id2_label = json.load(f)
        return id2_label

def id2_label(id2_label, id):
    return id2_label[id]

def display_cm(labels, predicted_classes):

    id2_label = load_id2label()
    ticks = []
    for i in range(27):
        ticks.append(id2_label[str(i)])
    
    class_report = classification_report(labels, predicted_classes)
    print("Classification Report:\n", class_report)

    print("\nConfusion Matrix:")
    cm = confusion_matrix(labels, predicted_classes, normalize='true')

    cm_display = np.round(cm, 2)

    plt.figure(figsize=(20, 10))
    sns.heatmap(cm_display, annot=True, fmt='.2f', cmap='viridis', xticklabels=ticks, yticklabels=ticks)
    plt.title("Confusion Matrix")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

### Generating Predictions from a Fine-Tuned BERT Model

This section demonstrates how to load a fine-tuned BERT checkpoint and generate predictions on a dataset:

1. **Load Model and Tokenizer**  
   - `AutoModelForSequenceClassification.from_pretrained(checkpoint_path)` loads the saved checkpoint of the fine-tuned model.  
   - `AutoTokenizer.from_pretrained(checkpoint_path)` ensures consistent tokenization.  
   - Move the model to the appropriate device (`CPU` or `GPU`) and set it to evaluation mode with `model.eval()`.

2. **Prediction Function**  
   - `get_predictions(model, dataloader, device)` iterates over a dataloader and computes predictions without updating model weights (`torch.no_grad()`).  
   - Extracts `input_ids` and `attention_mask` from the batch and optionally handles other modalities like `images`.  
   - Computes logits via the model and selects the predicted class using `torch.argmax`.  
   - Collects all true labels and predicted classes into NumPy arrays for evaluation.

This function provides a ready-to-use way to obtain predictions for computing metrics like accuracy, F1-score, and confusion matrices on any dataset.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Example for a BERT classifier
checkpoint_path = "./birdy-optimized/checkpoint-12576"  # replace with your actual checkpoint folder

model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

model.to(device)
model.eval()

import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


def get_predictions(model, dataloader, device):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            images = batch['images'].to(device) if 'images' in batch else None

            # Forward pass
            if images is not None:
                logits = model(input_ids, attention_mask, images)
            else:
                # For HF models, access the logits
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits

            preds = torch.argmax(logits, dim=-1)

            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    labels = np.concatenate(all_labels)
    predicted_classes = np.concatenate(all_preds)
    return labels, predicted_classes

### Evaluating the Model on the Test Set

This snippet sets up a DataLoader for the test dataset, generates predictions, and visualizes the results:

1. **Test DataLoader**  
   - `DataLoader` is used to create batches from the tokenized test dataset.  
   - `default_data_collator` ensures each batch is returned as a dictionary of tensors, compatible with the model.

2. **Generate Predictions**  
   - `get_predictions(model, test_loader, device)` runs the model on the test set and collects true labels and predicted classes.

3. **Display Metrics and Confusion Matrix**  
   - `display_cm(labels, predicted_classes)` prints a detailed classification report and shows a normalized confusion matrix using a heatmap for easy visualization.  

This allows a complete evaluation of the fine-tuned model on unseen data, helping identify class-wise performance and potential areas for improvement.

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

test_loader = DataLoader(
    tokenized_datasets["test"],
    batch_size=16,
    collate_fn=default_data_collator  # important to get batch as dict of tensors
)

labels, predicted_classes = get_predictions(model, test_loader, device)
display_cm(labels, predicted_classes)